In [1]:
import sys
import pandas as pd
import numpy as np
import scipy.sparse as sparse
import implicit

# 1. Load the time-split data
sys.path.append('../src')
from data_prep import load_and_split_data
train_data, test_data, orders = load_and_split_data()

# 2. Build the User-Product Matrix
print("Building the user-product matrix...")
# Count how many times each user bought each product
user_product_counts = train_data.groupby(['user_id', 'product_id']).size().reset_index(name='purchase_count')

# Map IDs to sequential indices for the math model
users = user_product_counts['user_id'].astype('category')
items = user_product_counts['product_id'].astype('category')

user_indices = users.cat.codes
item_indices = items.cat.codes
counts = user_product_counts['purchase_count'].astype(np.float32)

# Create a sparse matrix (saves RAM by only storing cells with actual purchases)
user_item_matrix = sparse.csr_matrix((counts, (user_indices, item_indices)))

print(f"Matrix shape: {user_item_matrix.shape[0]:,} users and {user_item_matrix.shape[1]:,} products.")

# 3. Train the ALS Model
print("Training the ALS Collaborative Filtering model (this takes a moment)...")
# factors = complexity of patterns, iterations = training loops
model = implicit.als.AlternatingLeastSquares(factors=50, iterations=15, regularization=0.01, random_state=42)

# Fit the model
model.fit(user_item_matrix)
print("ALS Model trained successfully!")

Loading datasets...
Merging data to create time-based Train/Test splits...
Split complete!
Building the user-product matrix...
Matrix shape: 206,209 users and 49,677 products.
Training the ALS Collaborative Filtering model (this takes a moment)...


c:\Users\GARV DHINGRA\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: OpenBLAS is configured to use 12 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()


  0%|          | 0/15 [00:00<?, ?it/s]

ALS Model trained successfully!


In [2]:
import pandas as pd

# 1. Load the product dictionary so we can read the actual names
products = pd.read_csv('../data/raw/products.csv', usecols=['product_id', 'product_name'])
product_dict = dict(zip(products['product_id'], products['product_name']))

# 2. Pick a specific user to test (we will use the very first user in our matrix)
matrix_user_index = 0
real_user_id = users.cat.categories[matrix_user_index]

print(f"Generating personalized recommendations for User ID: {real_user_id}...")

# 3. Ask the ALS model for top 10 recommendations
# filter_already_liked_items=True forces the model to suggest brand NEW discoveries 
# rather than just repeating their old habits.
recommended_indices, scores = model.recommend(
    matrix_user_index, 
    user_item_matrix[matrix_user_index], 
    N=10, 
    filter_already_liked_items=True 
)

# 4. Convert the math indices back to real product IDs
recommended_product_ids = items.cat.categories[recommended_indices]

print("\n✨ Top 10 New Product Discoveries (ALS Predictions):")
for rank, (prod_id, score) in enumerate(zip(recommended_product_ids, scores), 1):
    name = product_dict.get(prod_id, 'Unknown Product')
    print(f"{rank}. {name} (Confidence: {score:.3f})")

# 5. Print their past history so we can judge if the predictions make logical sense
past_history = train_data[train_data['user_id'] == real_user_id]['product_id'].value_counts().head(10).index
print("\n🛒 For context, their most frequently bought past items are:")
for past_id in past_history:
    print(f"- {product_dict.get(past_id, 'Unknown Product')}")

Generating personalized recommendations for User ID: 1...

✨ Top 10 New Product Discoveries (ALS Predictions):
1. Clementines (Confidence: 0.551)
2. Trail Mix (Confidence: 0.544)
3. Extra Fancy Unsalted Mixed Nuts (Confidence: 0.477)
4. Crunchy Oats 'n Honey Granola Bars (Confidence: 0.472)
5. Sparkling Mineral Water (Confidence: 0.441)
6. Reduced Fat 2% Milk (Confidence: 0.433)
7. Apples (Confidence: 0.428)
8. Mozzarella String Cheese (Confidence: 0.427)
9. Popcorn (Confidence: 0.410)
10. Mixed Fruit Fruit Snacks (Confidence: 0.407)

🛒 For context, their most frequently bought past items are:
- Soda
- Original Beef Jerky
- Pistachios
- Organic String Cheese
- Cinnamon Toast Crunch
- Zero Calorie Cola
- Bag of Organic Bananas
- XL Pick-A-Size Paper Towel Rolls
- Organic Half & Half
- Aged White Cheddar Popcorn


In [3]:
import os

# Create the app directory if it doesn't exist
os.makedirs('../app', exist_ok=True)

# Save the trained ALS model
model.save('../app/als_model.npz')
print("Model successfully saved for production!")

Model successfully saved for production!


In [4]:
import sys
print(sys.executable)

c:\Users\GARV DHINGRA\AppData\Local\Python\pythoncore-3.14-64\python.exe
